# Feature Engineering (Fixed) -- Group 8 Flood Risk Prediction

## What changed vs the original notebook

**ERA5 section: full rewrite.** The original computed rolling and percentile features on already-collapsed regional time series, which produced just two rows of features (one per region). Every terrain pixel in a region ended up with identical weather values. The fixed version loads the spatially-resolved ERA5 parquets produced by `data_cleaning_fixed.ipynb` and computes the same features **per ERA5 spatial pixel**, yielding ~414 rows of weather features for Severn and ~234 for Northumbria. A new spatial z-score family of features is also added, matching the suggestion on slide 24 of the project brief.

**Terrain section: mostly unchanged.** Inspection cells, the chunked flattening loop, and the `is_waterway` / `imd` / `clc_type` / `waw` fixes are kept. One change: the flattening loop now also computes the nearest ERA5 pixel index for each terrain pixel using a KDTree, and stores it as `era5_pixel_idx` in the terrain parquet. This lets the model look up weather features by index at training time without baking 100+ redundant weather columns into the multi-gigabyte terrain file.

## Inputs

- `era5_severn_daily_spatial.parquet` and `era5_northumbria_daily_spatial.parquet` (NEW format from the fixed cleaning notebook)
- `terrain_severn_clean.nc` and `terrain_northumbria_clean.nc` (unchanged)

## Outputs

| File | Format | Rows | Note |
|---|---|---|---|
| `era5_severn_pixel_features.parquet` | Parquet | ~414 | One row per ERA5 pixel, ~110 weather features |
| `era5_northumbria_pixel_features.parquet` | Parquet | ~234 | One row per ERA5 pixel, ~110 weather features |
| `severn_terrain_ready.parquet` | Parquet | ~35M | Per-pixel terrain features + `era5_pixel_idx` lookup column |
| `northumbria_terrain_ready.parquet` | Parquet | ~21M | Per-pixel terrain features + `era5_pixel_idx` lookup column |

## How to join at training time

```python
terrain = pd.read_parquet("severn_terrain_ready.parquet")
weather = pd.read_parquet("era5_severn_pixel_features.parquet")
# weather is indexed 0..N-1, matching the values stored in terrain["era5_pixel_idx"]
merged = terrain.merge(weather, left_on="era5_pixel_idx", right_index=True)
```


## ERA5 weather features (per spatial pixel)

In [2]:
# ── Step 1: Load and inspect the new spatially-resolved ERA5 parquets ─────
import pandas as pd
import numpy as np
from pathlib import Path

_cwd = Path.cwd()
DATA_DIR = _cwd / "Data" if (_cwd / "Data").exists() else (_cwd.parent / "Data").resolve()
OUT_DIR = DATA_DIR / "cleaned"

path_severn      = OUT_DIR / "era5_severn_daily_spatial.parquet"
path_northumbria = OUT_DIR / "era5_northumbria_daily_spatial.parquet"

df_sev = pd.read_parquet(path_severn)
df_nor = pd.read_parquet(path_northumbria)

for name, df in [("SEVERN", df_sev), ("NORTHUMBRIA", df_nor)]:
    print(f"\n{'='*60}\n  {name}\n{'='*60}")
    print(f"Shape: {df.shape}")
    print(f"Columns ({len(df.columns)}): {list(df.columns)}")
    n_pixels = df[["proj_y", "proj_x"]].drop_duplicates().shape[0]
    n_days   = df["valid_time"].nunique()
    print(f"Unique ERA5 pixels: {n_pixels}")
    print(f"Unique days:        {n_days}")
    print(f"Expected total rows = {n_pixels * n_days}, actual = {len(df)}")
    print(f"Date range: {df['valid_time'].min()} to {df['valid_time'].max()}")
    print(f"Null counts:\n{df.isnull().sum()}")



  SEVERN
Shape: (774436, 18)
Columns (18): ['valid_time', 'proj_y', 'proj_x', 'u10_mean', 'v10_mean', 'd2m_mean', 't2m_mean', 'sp_mean', 'swvl1_mean', 'u10_max', 'v10_max', 'd2m_max', 't2m_max', 'sp_max', 'swvl1_max', 'sro', 'tp', 'region']
Unique ERA5 pixels: 212
Unique days:        3653
Expected total rows = 774436, actual = 774436
Date range: 2015-01-01 00:00:00 to 2024-12-31 00:00:00
Null counts:
valid_time    0
proj_y        0
proj_x        0
u10_mean      0
v10_mean      0
d2m_mean      0
t2m_mean      0
sp_mean       0
swvl1_mean    0
u10_max       0
v10_max       0
d2m_max       0
t2m_max       0
sp_max        0
swvl1_max     0
sro           0
tp            0
region        0
dtype: int64

  NORTHUMBRIA
Shape: (434707, 18)
Columns (18): ['valid_time', 'proj_y', 'proj_x', 'u10_mean', 'v10_mean', 'd2m_mean', 't2m_mean', 'sp_mean', 'swvl1_mean', 'u10_max', 'v10_max', 'd2m_max', 't2m_max', 'sp_max', 'swvl1_max', 'sro', 'tp', 'region']
Unique ERA5 pixels: 119
Unique days:        365

In [3]:
# ── Step 1b: Filter to valid ERA5 pixels (non-null, within region boundary) ─
# Drop coordinate/metadata columns that are not weather variables.
DROP_COLS = ["spatial_ref", "number"]

def filter_valid_pixels(df):
    """
    Drop non-weather columns and remove ERA5 pixels that are entirely null
    across all weather variables. These are grid cells inside the bounding
    box but outside the actual catchment boundary.
    """
    # Drop metadata columns
    df = df.drop(columns=[c for c in DROP_COLS if c in df.columns])

    # Identify weather variable columns (everything except coordinates)
    coord_cols = ["valid_time", "proj_y", "proj_x", "region"]
    weather_cols = [c for c in df.columns if c not in coord_cols]

    # A pixel is invalid if ALL its weather values are null across all days.
    # Compute per-pixel null fractions to find always-null pixels.
    pixel_null_frac = (
        df.groupby(["proj_y", "proj_x"])[weather_cols[0]]
        .apply(lambda s: s.isnull().all())
    )
    invalid_pixels = pixel_null_frac[pixel_null_frac].reset_index()[["proj_y", "proj_x"]]
    n_invalid = len(invalid_pixels)

    df_valid = df.merge(invalid_pixels, on=["proj_y", "proj_x"], how="left", indicator=True)
    df_valid = df_valid[df_valid["_merge"] == "left_only"].drop(columns=["_merge"])

    print(f"  Dropped {n_invalid} always-null ERA5 pixels "
          f"({100*n_invalid/(n_invalid + df_valid[['proj_y','proj_x']].drop_duplicates().shape[0]):.1f}% of bounding box)")
    print(f"  Valid pixels remaining: {df_valid[['proj_y','proj_x']].drop_duplicates().shape[0]}")
    print(f"  Remaining nulls in weather cols: {df_valid[weather_cols].isnull().sum().sum()}")
    return df_valid.reset_index(drop=True)

print("Filtering Severn ERA5 ...")
df_sev = filter_valid_pixels(df_sev)

print("\nFiltering Northumbria ERA5 ...")
df_nor = filter_valid_pixels(df_nor)

Filtering Severn ERA5 ...
  Dropped 0 always-null ERA5 pixels (0.0% of bounding box)
  Valid pixels remaining: 212
  Remaining nulls in weather cols: 0

Filtering Northumbria ERA5 ...
  Dropped 0 always-null ERA5 pixels (0.0% of bounding box)
  Valid pixels remaining: 119
  Remaining nulls in weather cols: 0


In [4]:
# ── Step 2: Helper to pivot a variable to (date, pixel) shape ─────────────
# A wide layout (rows = dates, cols = pixel_id) lets us use pandas vectorized
# rolling / quantile operations across all pixels at once, rather than looping.

def make_pixel_id(df):
    """Stable integer pixel id derived from sorted unique (y, x) pairs."""
    coords = (
        df[["proj_y", "proj_x"]]
        .drop_duplicates()
        .sort_values(["proj_y", "proj_x"])
        .reset_index(drop=True)
    )
    coords["pixel_id"] = np.arange(len(coords), dtype=np.int32)
    return df.merge(coords, on=["proj_y", "proj_x"], how="left"), coords

def to_wide(df, var):
    """Return a wide dataframe: index = valid_time, columns = pixel_id."""
    return (
        df.pivot(index="valid_time", columns="pixel_id", values=var)
          .sort_index()
    )

# Build pixel ids for both regions
df_sev, coords_sev = make_pixel_id(df_sev)
df_nor, coords_nor = make_pixel_id(df_nor)

print(f"Severn:      {len(coords_sev)} pixel ids assigned")
print(f"Northumbria: {len(coords_nor)} pixel ids assigned")


Severn:      212 pixel ids assigned
Northumbria: 119 pixel ids assigned


In [5]:
# ── Step 3: Per-pixel rolling-window features for tp and sro ──────────────
# For each pixel, compute the maximum rolling sum over 5, 10, 15 day windows.
# This captures the worst sustained rainfall / runoff event that specific
# pixel experienced over the full 10-year period.

def rolling_max_per_pixel(wide, window):
    """
    wide: DataFrame indexed by date, columns = pixel_id.
    Returns a Series indexed by pixel_id giving the max rolling sum.
    """
    return wide.rolling(window=window, min_periods=window).sum().max(axis=0)

def build_rolling_features(df, var_list, windows):
    feats = {}
    for var in var_list:
        wide = to_wide(df, var)
        for w in windows:
            feats[f"max_rolling_{w}d_{var}"] = rolling_max_per_pixel(wide, w)
    return pd.DataFrame(feats)

rolling_vars = ["tp", "sro"]
rolling_windows = [5, 10, 15]

roll_sev = build_rolling_features(df_sev, rolling_vars, rolling_windows)
roll_nor = build_rolling_features(df_nor, rolling_vars, rolling_windows)

print(f"Severn rolling features:      {roll_sev.shape}")
print(f"Northumbria rolling features: {roll_nor.shape}")
print(f"\nSevern sample (first 3 pixels):\n{roll_sev.head(3)}")


Severn rolling features:      (212, 6)
Northumbria rolling features: (119, 6)

Severn sample (first 3 pixels):
          max_rolling_5d_tp  max_rolling_10d_tp  max_rolling_15d_tp  \
pixel_id                                                              
0                  0.099581            0.111326            0.129849   
1                  0.100921            0.112600            0.126666   
2                  0.095217            0.111081            0.134715   

          max_rolling_5d_sro  max_rolling_10d_sro  max_rolling_15d_sro  
pixel_id                                                                
0                   0.006596             0.006669             0.009010  
1                   0.006376             0.007995             0.009213  
2                   0.010772             0.010876             0.012864  


In [6]:
# ── Step 4: Per-pixel soil moisture extremes ──────────────────────────────
# 7-day rolling average of soil moisture (swvl1_mean), then max (peak
# saturation) and min (peak depletion) per pixel. Also overall mean/std.

def build_soil_features(df):
    feats = {}
    wide_mean = to_wide(df, "swvl1_mean")
    roll7 = wide_mean.rolling(window=7, min_periods=7).mean()
    feats["max_rolling_7d_swvl1_mean"] = roll7.max(axis=0)
    feats["min_rolling_7d_swvl1_mean"] = roll7.min(axis=0)
    feats["swvl1_mean_overall"] = wide_mean.mean(axis=0)
    feats["swvl1_std_overall"]  = wide_mean.std(axis=0)

    # Also do the daily-max soil moisture if present
    if "swvl1_max" in df.columns:
        wide_max = to_wide(df, "swvl1_max")
        roll7m = wide_max.rolling(window=7, min_periods=7).mean()
        feats["max_rolling_7d_swvl1_max"] = roll7m.max(axis=0)
        feats["min_rolling_7d_swvl1_max"] = roll7m.min(axis=0)

    return pd.DataFrame(feats)

soil_sev = build_soil_features(df_sev)
soil_nor = build_soil_features(df_nor)

print(f"Severn soil features:      {soil_sev.shape}")
print(f"Northumbria soil features: {soil_nor.shape}")
print(f"\nSevern sample (first 3 pixels):\n{soil_sev.head(3)}")


Severn soil features:      (212, 6)
Northumbria soil features: (119, 6)

Severn sample (first 3 pixels):
          max_rolling_7d_swvl1_mean  min_rolling_7d_swvl1_mean  \
pixel_id                                                         
0                          0.476059                   0.238414   
1                          0.465522                   0.220626   
2                          0.414860                   0.151051   

          swvl1_mean_overall  swvl1_std_overall  max_rolling_7d_swvl1_max  \
pixel_id                                                                    
0                   0.415778           0.052940                  0.494884   
1                   0.403028           0.055097                  0.484569   
2                   0.345642           0.058288                  0.433934   

          min_rolling_7d_swvl1_max  
pixel_id                            
0                         0.239332  
1                         0.222421  
2                         0.15

In [7]:
# ── Step 5: Per-pixel percentile features for tp and sro ──────────────────
# For each pixel, compute the 50/90/95/99 percentiles, plus overall mean/std.

def build_percentile_features(df, var_list, percentiles):
    feats = {}
    for var in var_list:
        wide = to_wide(df, var)
        # Quantile across the time axis returns a frame of (quantiles, pixel_id)
        qs = wide.quantile(q=[p/100 for p in percentiles], axis=0).T
        qs.columns = [f"{var}_p{p}" for p in percentiles]
        feats[f"__{var}_qs"] = qs
        feats[f"{var}_overall_mean"] = wide.mean(axis=0)
        feats[f"{var}_overall_std"]  = wide.std(axis=0)

    # Stitch together
    out = pd.DataFrame()
    for k, v in feats.items():
        if k.startswith("__"):
            out = pd.concat([out, v], axis=1)
        else:
            out[k] = v
    return out

pct_vars = ["tp", "sro"]
percentiles = [50, 90, 95, 99]

pct_sev = build_percentile_features(df_sev, pct_vars, percentiles)
pct_nor = build_percentile_features(df_nor, pct_vars, percentiles)

print(f"Severn percentile features:      {pct_sev.shape}")
print(f"Northumbria percentile features: {pct_nor.shape}")
print(f"\nSevern sample (first 3 pixels):\n{pct_sev.head(3)}")


Severn percentile features:      (212, 12)
Northumbria percentile features: (119, 12)

Severn sample (first 3 pixels):
            tp_p50    tp_p90    tp_p95    tp_p99  tp_overall_mean  \
pixel_id                                                            
0         0.000546  0.007885  0.010958  0.018489         0.002434   
1         0.000519  0.007766  0.010961  0.018242         0.002412   
2         0.000585  0.008020  0.011224  0.018882         0.002496   

          tp_overall_std   sro_p50   sro_p90   sro_p95   sro_p99  \
pixel_id                                                           
0               0.004107  0.000005  0.000151  0.000372  0.001299   
1               0.004105  0.000005  0.000142  0.000362  0.001353   
2               0.004161  0.000005  0.000126  0.000281  0.001387   

          sro_overall_mean  sro_overall_std  
pixel_id                                     
0                 0.000079         0.000259  
1                 0.000079         0.000271  
2         

In [8]:
# ── Step 6: Per-pixel seasonal extremes ───────────────────────────────────
# For each meteorological season (DJF, MAM, JJA, SON), compute mean / max /
# std of tp, sro, and swvl1_mean per pixel.

SEASON_MAP = {12: "DJF", 1: "DJF", 2: "DJF",
              3: "MAM", 4: "MAM", 5: "MAM",
              6: "JJA", 7: "JJA", 8: "JJA",
              9: "SON", 10: "SON", 11: "SON"}

def build_seasonal_features(df, var_list):
    feats_frames = []
    df = df.copy()
    df["season"] = df["valid_time"].dt.month.map(SEASON_MAP)

    for var in var_list:
        # group by (pixel_id, season) and aggregate
        g = df.groupby(["pixel_id", "season"])[var].agg(["mean", "max", "std"])
        g = g.unstack("season")  # columns become (stat, season)
        # Flatten columns to "{var}_{season}_{stat}"
        g.columns = [f"{var}_{season}_{stat}" for stat, season in g.columns]
        feats_frames.append(g)

    return pd.concat(feats_frames, axis=1)

season_vars = ["tp", "sro", "swvl1_mean"]
seas_sev = build_seasonal_features(df_sev, season_vars)
seas_nor = build_seasonal_features(df_nor, season_vars)

print(f"Severn seasonal features:      {seas_sev.shape}")
print(f"Northumbria seasonal features: {seas_nor.shape}")
print(f"\nSevern sample (first 3 pixels, first 8 cols):\n{seas_sev.iloc[:3, :8]}")


Severn seasonal features:      (212, 36)
Northumbria seasonal features: (119, 36)

Severn sample (first 3 pixels, first 8 cols):
          tp_DJF_mean  tp_JJA_mean  tp_MAM_mean  tp_SON_mean  tp_DJF_max  \
pixel_id                                                                   
0            0.002395     0.002335     0.002227     0.002782    0.025446   
1            0.002366     0.002310     0.002209     0.002768    0.026331   
2            0.002505     0.002389     0.002244     0.002850    0.023692   

          tp_JJA_max  tp_MAM_max  tp_SON_max  
pixel_id                                      
0           0.030864    0.034237    0.038147  
1           0.030925    0.033870    0.038413  
2           0.031124    0.035345    0.036935  


In [9]:
# ── Step 7: Spatial z-score features (per slide 24 of the brief) ───────────
# For each day, normalize each pixel's value relative to that day's regional
# mean and std. This captures "how anomalous is this pixel relative to the
# rest of the region on any given day?" Then take time-aggregate summaries
# per pixel.
#
# Note on sro_spatial_z_max: surface runoff is extremely spatially
# concentrated on individual events, producing a tiny regional std and a
# massive z-score on certain days regardless of any threshold. This makes
# the max an unstable feature. It is dropped here in favor of the more
# robust p95, p99, and day-count features.

MIN_THRESHOLD = {"tp": 1e-4, "sro": 1e-5, "swvl1_mean": 0.0}

def build_spatial_z_features(df, var_list):
    feats = {}
    for var in var_list:
        wide = to_wide(df, var)
        daily_mean = wide.mean(axis=1)
        daily_std  = wide.std(axis=1).replace(0, np.nan)

        # Mask out near-dry days for precipitation and runoff to prevent
        # near-zero regional std from inflating z-scores artificially
        threshold = MIN_THRESHOLD.get(var, 0.0)
        valid_days = daily_mean >= threshold
        daily_std_masked = daily_std.where(valid_days)

        z = wide.sub(daily_mean, axis=0).div(daily_std_masked, axis=0)

        feats[f"{var}_spatial_z_mean"]    = z.mean(axis=0)
        # spatial_z_max is intentionally excluded for sro due to instability
        # from extreme spatial concentration events (see note above)
        if var != "sro":
            feats[f"{var}_spatial_z_max"] = z.max(axis=0)
        feats[f"{var}_spatial_z_p95"]     = z.quantile(0.95, axis=0)
        feats[f"{var}_spatial_z_p99"]     = z.quantile(0.99, axis=0)
        feats[f"{var}_n_days_z_gt_1"]     = (z > 1).sum(axis=0).astype("int32")
        feats[f"{var}_n_days_z_gt_2"]     = (z > 2).sum(axis=0).astype("int32")

    return pd.DataFrame(feats)

z_vars = ["tp", "sro", "swvl1_mean"]
z_sev = build_spatial_z_features(df_sev, z_vars)
z_nor = build_spatial_z_features(df_nor, z_vars)

print(f"Severn spatial-z features:      {z_sev.shape}")
print(f"Northumbria spatial-z features: {z_nor.shape}")
print(f"\nExpected columns: tp has max, sro does not, swvl1_mean has max")
print(f"Severn columns: {list(z_sev.columns)}")
print(f"\nSevern sample (first 3 pixels):\n{z_sev.head(3)}")

Severn spatial-z features:      (212, 17)
Northumbria spatial-z features: (119, 17)

Expected columns: tp has max, sro does not, swvl1_mean has max
Severn columns: ['tp_spatial_z_mean', 'tp_spatial_z_max', 'tp_spatial_z_p95', 'tp_spatial_z_p99', 'tp_n_days_z_gt_1', 'tp_n_days_z_gt_2', 'sro_spatial_z_mean', 'sro_spatial_z_p95', 'sro_spatial_z_p99', 'sro_n_days_z_gt_1', 'sro_n_days_z_gt_2', 'swvl1_mean_spatial_z_mean', 'swvl1_mean_spatial_z_max', 'swvl1_mean_spatial_z_p95', 'swvl1_mean_spatial_z_p99', 'swvl1_mean_n_days_z_gt_1', 'swvl1_mean_n_days_z_gt_2']

Severn sample (first 3 pixels):
          tp_spatial_z_mean  tp_spatial_z_max  tp_spatial_z_p95  \
pixel_id                                                          
0                  0.097341          7.000139          2.702468   
1                  0.071235          7.233906          2.777165   
2                  0.145916          4.399996          2.470032   

          tp_spatial_z_p99  tp_n_days_z_gt_1  tp_n_days_z_gt_2  \
pixe

In [10]:
# ── Step 8: Assemble all per-pixel ERA5 features and save ─────────────────
# Combine rolling + soil + percentile + seasonal + spatial-z feature frames.
# Attach the proj_y / proj_x coordinates so we can spatially join terrain
# pixels in the next section.

def assemble_features(coords, *frames, region_name):
    """
    coords: dataframe with columns [proj_y, proj_x, pixel_id]
    frames: feature dataframes indexed by pixel_id
    """
    combined = pd.concat(frames, axis=1)
    combined.index.name = "pixel_id"
    combined = combined.reset_index()
    out = coords.merge(combined, on="pixel_id", how="left")
    out["region"] = region_name

    # Order columns: identifiers first, then features
    id_cols = ["pixel_id", "proj_y", "proj_x", "region"]
    feat_cols = [c for c in out.columns if c not in id_cols]
    out = out[id_cols + feat_cols]
    # Downcast for parquet size
    for c in feat_cols:
        if out[c].dtype == "float64":
            out[c] = out[c].astype("float32")
    return out

era5_feats_sev = assemble_features(
    coords_sev, roll_sev, soil_sev, pct_sev, seas_sev, z_sev,
    region_name="severn"
)
era5_feats_nor = assemble_features(
    coords_nor, roll_nor, soil_nor, pct_nor, seas_nor, z_nor,
    region_name="northumbria"
)

print(f"Severn ERA5 features:      {era5_feats_sev.shape}")
print(f"Northumbria ERA5 features: {era5_feats_nor.shape}")
print(f"Severn null count:      {era5_feats_sev.isnull().sum().sum()}")
print(f"Northumbria null count: {era5_feats_nor.isnull().sum().sum()}")
print(f"\nFeature column names (first 20): {list(era5_feats_sev.columns[:20])}")

era5_feats_sev.to_parquet(OUT_DIR / "era5_severn_pixel_features.parquet",      index=False)
era5_feats_nor.to_parquet(OUT_DIR / "era5_northumbria_pixel_features.parquet", index=False)
print("\nSaved per-pixel ERA5 feature parquets.")


Severn ERA5 features:      (212, 81)
Northumbria ERA5 features: (119, 81)
Severn null count:      0
Northumbria null count: 0

Feature column names (first 20): ['pixel_id', 'proj_y', 'proj_x', 'region', 'max_rolling_5d_tp', 'max_rolling_10d_tp', 'max_rolling_15d_tp', 'max_rolling_5d_sro', 'max_rolling_10d_sro', 'max_rolling_15d_sro', 'max_rolling_7d_swvl1_mean', 'min_rolling_7d_swvl1_mean', 'swvl1_mean_overall', 'swvl1_std_overall', 'max_rolling_7d_swvl1_max', 'min_rolling_7d_swvl1_max', 'tp_p50', 'tp_p90', 'tp_p95', 'tp_p99']

Saved per-pixel ERA5 feature parquets.


## Terrain features (per-pixel flatten + spatial join to ERA5)

In [11]:
# ── Step 9: Inspect cleaned terrain ───────────────────────────────────────
import xarray as xr

path_sev = OUT_DIR / "terrain_severn_clean.nc"
path_nor = OUT_DIR / "terrain_northumbria_clean.nc"

for name, path in [("SEVERN", path_sev), ("NORTHUMBRIA", path_nor)]:
    ds = xr.open_dataset(path, chunks={"x": 2000, "y": 2000})
    print(f"\n{'='*60}\n  {name}\n{'='*60}")
    print(f"Dimensions: {dict(ds.dims)}")
    print(f"Variables:  {list(ds.data_vars)}")
    if "valid_pixel" in ds.data_vars:
        n_valid = ds["valid_pixel"].sum().compute().item()
        n_total = ds["valid_pixel"].size
        print(f"Valid pixels: {n_valid:,} / {n_total:,} ({100*n_valid/n_total:.1f}%)")
    ds.close()



  SEVERN
Dimensions: {'y': 10249, 'x': 8192}
Variables:  ['dtm', 'flow_acc', 'imd', 'waw', 'rciw', 'clc_type', 'risk_0_2m', 'risk_0_3m', 'risk_0_6m', 'risk_0_9m', 'risk_1_2m', 'log_flow_acc', 'dtm_zscore', 'valid_pixel']
Valid pixels: 35,278,130 / 83,959,808 (42.0%)

  NORTHUMBRIA
Dimensions: {'y': 8327, 'x': 5527}
Variables:  ['dtm', 'flow_acc', 'imd', 'waw', 'rciw', 'clc_type', 'risk_0_2m', 'risk_0_3m', 'risk_0_6m', 'risk_0_9m', 'risk_1_2m', 'log_flow_acc', 'dtm_zscore', 'valid_pixel']
Valid pixels: 21,391,824 / 46,023,329 (46.5%)


In [12]:
# ── Step 10: Full-range checks for terrain fixes (imd, rciw, clc_type) ────
print("="*60)
print("  PART A: Full imd range check")
print("="*60)
for name, path in [("SEVERN", path_sev), ("NORTHUMBRIA", path_nor)]:
    ds = xr.open_dataset(path, chunks={"x": 2000, "y": 2000})
    imd_min = ds["imd"].min().compute().item()
    imd_max = ds["imd"].max().compute().item()
    print(f"  {name} imd range: {imd_min} to {imd_max}")
    ds.close()

print(f"\n{'='*60}\n  PART B: rciw value distribution\n{'='*60}")
for name, path in [("SEVERN", path_sev), ("NORTHUMBRIA", path_nor)]:
    ds = xr.open_dataset(path, chunks={"x": 2000, "y": 2000})
    valid_mask = ds["valid_pixel"].values
    rciw_vals = ds["rciw"].values
    valid_rciw = rciw_vals[valid_mask]
    n_valid = len(valid_rciw)
    n_nan = np.isnan(valid_rciw).sum()
    non_nan = valid_rciw[~np.isnan(valid_rciw)]
    print(f"\n  {name} (valid pixels only):")
    print(f"    Total valid pixels: {n_valid:,}")
    print(f"    rciw NaN: {n_nan:,} ({100*n_nan/n_valid:.2f}%)")
    if len(non_nan) > 0:
        unique, counts = np.unique(non_nan, return_counts=True)
        for u, c in zip(unique, counts):
            print(f"    rciw={int(u)}: {c:,} ({100*c/n_valid:.3f}%)")
    ds.close()

print(f"\n{'='*60}\n  PART C: clc_type unique values\n{'='*60}")
for name, path in [("SEVERN", path_sev), ("NORTHUMBRIA", path_nor)]:
    ds = xr.open_dataset(path, chunks={"x": 2000, "y": 2000})
    valid_mask = ds["valid_pixel"].values
    clc_vals = ds["clc_type"].values[valid_mask]
    clc_valid = clc_vals[~np.isnan(clc_vals)]
    unique = np.unique(clc_valid)
    print(f"  {name}: {len(unique)} classes: {sorted(unique.astype(int).tolist())}")
    ds.close()


  PART A: Full imd range check
  SEVERN imd range: 0.0 to 100.0
  NORTHUMBRIA imd range: 0.0 to 255.0

  PART B: rciw value distribution

  SEVERN (valid pixels only):
    Total valid pixels: 35,278,130
    rciw NaN: 35,138,961 (-22.14%)
    rciw=2: 95,767 (0.271%)
    rciw=3: 43,402 (0.123%)


C:\Users\xinra\AppData\Local\Temp\ipykernel_39848\786325810.py:23: RuntimeWarning: overflow encountered in scalar multiply
  print(f"    rciw NaN: {n_nan:,} ({100*n_nan/n_valid:.2f}%)")



  NORTHUMBRIA (valid pixels only):
    Total valid pixels: 21,391,824
    rciw NaN: 21,268,542 (99.42%)
    rciw=2: 97,325 (0.455%)
    rciw=3: 25,957 (0.121%)

  PART C: clc_type unique values
  SEVERN: 31 classes: [111, 112, 121, 122, 123, 124, 131, 132, 133, 141, 142, 211, 222, 231, 242, 243, 311, 312, 313, 321, 322, 324, 333, 411, 412, 421, 423, 511, 512, 522, 523]
  NORTHUMBRIA: 30 classes: [111, 112, 121, 122, 123, 124, 131, 132, 133, 141, 142, 211, 231, 243, 311, 312, 313, 321, 322, 324, 332, 333, 411, 412, 421, 423, 511, 512, 522, 523]


In [13]:
# ── Step 11: Flatten terrain + apply fixes + add era5_pixel_idx (FIXED) ──
# Same chunked flattening as before, with the addition of a KDTree-based
# spatial join that records each terrain pixel's nearest ERA5 pixel id.
# Storing only the integer index (4 bytes per pixel) avoids ballooning the
# terrain parquet by the ~110 weather feature columns.

import gc
from scipy.spatial import cKDTree

terrain_paths = {
    "severn":      OUT_DIR / "terrain_severn_clean.nc",
    "northumbria": OUT_DIR / "terrain_northumbria_clean.nc",
}
era5_feature_paths = {
    "severn":      OUT_DIR / "era5_severn_pixel_features.parquet",
    "northumbria": OUT_DIR / "era5_northumbria_pixel_features.parquet",
}

terrain_vars = [
    "dtm_zscore", "log_flow_acc", "imd", "waw", "rciw", "clc_type",
    "risk_0_2m", "risk_0_3m", "risk_0_6m", "risk_0_9m", "risk_1_2m"
]

CHUNK_SIZE = 500  # rows of the y-axis per chunk

for region, path in terrain_paths.items():
    print(f"\n{'='*60}\n  Processing {region.upper()}\n{'='*60}")

    # Build a KDTree on ERA5 pixel coordinates for this region.
    era5_feats = pd.read_parquet(era5_feature_paths[region])
    era5_coords = era5_feats[["proj_y", "proj_x"]].values.astype(np.float32)
    era5_pixel_ids = era5_feats["pixel_id"].values.astype(np.int32)
    tree = cKDTree(era5_coords)
    print(f"  Built KDTree on {len(era5_coords)} ERA5 pixel centroids")

    ds = xr.open_dataset(path)
    ny, nx = ds.sizes["y"], ds.sizes["x"]
    x_vals = ds.coords["x"].values
    print(f"  Terrain grid: {ny} x {nx}, chunk size {CHUNK_SIZE} y-rows")

    out_path = OUT_DIR / f"{region}_terrain_ready.parquet"
    all_chunks = []
    total_valid = 0

    for y_start in range(0, ny, CHUNK_SIZE):
        y_end = min(y_start + CHUNK_SIZE, ny)
        chunk = ds.isel(y=slice(y_start, y_end))

        vp = chunk["valid_pixel"].values
        n_valid = vp.sum()
        if n_valid == 0:
            continue

        y_chunk = chunk.coords["y"].values
        y_grid, x_grid = np.meshgrid(y_chunk, x_vals, indexing="ij")

        proj_y = y_grid[vp].astype(np.float32)
        proj_x = x_grid[vp].astype(np.float32)

        data = {"proj_y": proj_y, "proj_x": proj_x}
        for var in terrain_vars:
            data[var] = chunk[var].values[vp]

        df_chunk = pd.DataFrame(data)
        del data, y_grid, x_grid, vp
        gc.collect()

        # Apply fixes (rciw -> is_waterway, clip imd, fill clc/waw NaN)
        df_chunk["is_waterway"] = np.where(
            np.isnan(df_chunk["rciw"]), 0, 1
        ).astype(np.int8)
        df_chunk.drop(columns=["rciw"], inplace=True)
        df_chunk["imd"]          = df_chunk["imd"].clip(0, 100).astype(np.float32)
        df_chunk["clc_type"]     = df_chunk["clc_type"].fillna(0).astype(np.int16)
        df_chunk["waw"]          = df_chunk["waw"].fillna(0).astype(np.int8)
        df_chunk["dtm_zscore"]   = df_chunk["dtm_zscore"].astype(np.float32)
        df_chunk["log_flow_acc"] = df_chunk["log_flow_acc"].astype(np.float32)

        # Spatial join: nearest ERA5 pixel per terrain pixel
        terrain_xy = np.column_stack([proj_y, proj_x])
        _, nn_idx = tree.query(terrain_xy, k=1, workers=-1)
        df_chunk["era5_pixel_idx"] = era5_pixel_ids[nn_idx].astype(np.int32)

        all_chunks.append(df_chunk)
        total_valid += len(df_chunk)

        if y_start % 5000 == 0:
            print(f"    y={y_start}-{y_end}: +{n_valid:,} pixels "
                  f"(cumulative: {total_valid:,})")

        del df_chunk
        gc.collect()

    ds.close()

    print(f"  Concatenating {len(all_chunks)} chunks ...")
    df = pd.concat(all_chunks, ignore_index=True)
    del all_chunks
    gc.collect()

    print(f"  Final shape: {df.shape}")
    null_counts = df.isnull().sum()
    print(f"  Nulls: {null_counts.sum()}")
    print(f"  Memory: {df.memory_usage(deep=True).sum() / 1e9:.2f} GB")
    print(f"  imd range: {df['imd'].min()} to {df['imd'].max()}")
    print(f"  is_waterway counts: 0={int((df['is_waterway']==0).sum()):,}, "
          f"1={int((df['is_waterway']==1).sum()):,}")
    print(f"  era5_pixel_idx range: {df['era5_pixel_idx'].min()} to {df['era5_pixel_idx'].max()}")
    print(f"  Unique era5_pixel_idx used: {df['era5_pixel_idx'].nunique()} / {len(era5_coords)}")
    print(f"  Columns: {list(df.columns)}")

    df.to_parquet(out_path, index=False)
    print(f"  Saved to: {out_path}")
    del df
    gc.collect()



  Processing SEVERN
  Built KDTree on 212 ERA5 pixel centroids
  Terrain grid: 10249 x 8192, chunk size 500 y-rows
    y=0-500: +349,071 pixels (cumulative: 349,071)
    y=5000-5500: +3,041,809 pixels (cumulative: 16,815,658)
    y=10000-10249: +319,629 pixels (cumulative: 35,278,130)
  Concatenating 21 chunks ...
  Final shape: (35278130, 14)
  Nulls: 0
  Memory: 1.16 GB
  imd range: 0.0 to 100.0
  is_waterway counts: 0=35,138,961, 1=139,169
  era5_pixel_idx range: 0 to 211
  Unique era5_pixel_idx used: 212 / 212
  Columns: ['proj_y', 'proj_x', 'dtm_zscore', 'log_flow_acc', 'imd', 'waw', 'clc_type', 'risk_0_2m', 'risk_0_3m', 'risk_0_6m', 'risk_0_9m', 'risk_1_2m', 'is_waterway', 'era5_pixel_idx']
  Saved to: D:\ESCP\T3\Hackathon\flood-risk-hackathon\Data\cleaned\severn_terrain_ready.parquet

  Processing NORTHUMBRIA
  Built KDTree on 119 ERA5 pixel centroids
  Terrain grid: 8327 x 5527, chunk size 500 y-rows
    y=0-500: +298,214 pixels (cumulative: 298,214)
    y=5000-5500: +1,786,28

In [14]:
# ── Step 12: Final validation ─────────────────────────────────────────────
df_sev_terrain = pd.read_parquet(OUT_DIR / "severn_terrain_ready.parquet")
df_nor_terrain = pd.read_parquet(OUT_DIR / "northumbria_terrain_ready.parquet")
df_sev_era5    = pd.read_parquet(OUT_DIR / "era5_severn_pixel_features.parquet")
df_nor_era5    = pd.read_parquet(OUT_DIR / "era5_northumbria_pixel_features.parquet")

print("="*60)
print("  FILE SUMMARY")
print("="*60)
print(f"  severn_terrain_ready.parquet:           {df_sev_terrain.shape}")
print(f"  northumbria_terrain_ready.parquet:      {df_nor_terrain.shape}")
print(f"  era5_severn_pixel_features.parquet:     {df_sev_era5.shape}")
print(f"  era5_northumbria_pixel_features.parquet:{df_nor_era5.shape}")

# CHECK 1: No nulls
print(f"\n{'='*60}\n  CHECK 1: Null counts\n{'='*60}")
for name, df in [
    ("Severn terrain",      df_sev_terrain),
    ("Northumbria terrain", df_nor_terrain),
    ("Severn ERA5",         df_sev_era5),
    ("Northumbria ERA5",    df_nor_era5),
]:
    n = df.isnull().sum().sum()
    status = "PASS" if n == 0 else "FAIL"
    print(f"  {name}: {n} nulls ({status})")

# CHECK 2: Risk class distribution
print(f"\n{'='*60}\n  CHECK 2: Target class distribution\n{'='*60}")
risk_cols = ["risk_0_2m", "risk_0_3m", "risk_0_6m", "risk_0_9m", "risk_1_2m"]
for name, df in [("SEVERN (train)", df_sev_terrain), ("NORTHUMBRIA (test)", df_nor_terrain)]:
    print(f"\n  {name}:")
    for col in risk_cols:
        counts = df[col].value_counts().sort_index()
        total = len(df)
        dist = ", ".join([f"{k}:{100*v/total:.1f}%" for k, v in counts.items()])
        print(f"    {col}: {dist}")

# CHECK 3: Feature ranges across regions
print(f"\n{'='*60}\n  CHECK 3: Feature range comparison\n{'='*60}")
feature_cols = ["dtm_zscore", "log_flow_acc", "imd", "waw", "clc_type", "is_waterway"]
for col in feature_cols:
    s_min, s_max = df_sev_terrain[col].min(), df_sev_terrain[col].max()
    n_min, n_max = df_nor_terrain[col].min(), df_nor_terrain[col].max()
    print(f"  {col:20s}  Severn=[{s_min:.2f}, {s_max:.2f}]  "
          f"Northumbria=[{n_min:.2f}, {n_max:.2f}]")

# CHECK 4: ERA5 pixel index integrity (every terrain index must map to a real ERA5 row)
print(f"\n{'='*60}\n  CHECK 4: era5_pixel_idx integrity\n{'='*60}")
for region, terrain, era5 in [
    ("Severn",      df_sev_terrain, df_sev_era5),
    ("Northumbria", df_nor_terrain, df_nor_era5),
]:
    n_unique_used = terrain["era5_pixel_idx"].nunique()
    valid_ids = set(era5["pixel_id"].values.tolist())
    used_ids  = set(terrain["era5_pixel_idx"].unique().tolist())
    missing = used_ids - valid_ids
    print(f"  {region}: terrain references {n_unique_used} / {len(era5)} ERA5 pixels. "
          f"Missing references: {len(missing)} ({'PASS' if len(missing)==0 else 'FAIL'})")

# CHECK 5: Spatial weather variation (the whole point of the fix)
print(f"\n{'='*60}\n  CHECK 5: Within-region spatial weather variation\n{'='*60}")
for region, era5 in [("Severn", df_sev_era5), ("Northumbria", df_nor_era5)]:
    if "tp_p99" in era5.columns:
        spread = era5["tp_p99"].max() - era5["tp_p99"].min()
        print(f"  {region} tp_p99 spread across pixels: {spread:.6f} "
              f"(min={era5['tp_p99'].min():.6f}, max={era5['tp_p99'].max():.6f})")
        if spread > 0:
            print(f"    PASS: weather varies spatially within {region}")
        else:
            print(f"    FAIL: weather is uniform within {region}")

# CHECK 6: Demonstrate the merge that the modeling step will perform
print(f"\n{'='*60}\n  CHECK 6: Example merge (Severn, 1000 rows)\n{'='*60}")
sample = df_sev_terrain.head(1000).copy()
weather_lookup = df_sev_era5.set_index("pixel_id")
merged = sample.merge(
    weather_lookup, left_on="era5_pixel_idx", right_index=True,
    suffixes=("", "_weather"),
)
print(f"  Sample terrain rows: {len(sample)}")
print(f"  After merge: {len(merged)} rows, {merged.shape[1]} columns")
print(f"  New weather columns present: "
      f"{[c for c in merged.columns if c.startswith('max_rolling_5d_tp')]}")

del df_sev_terrain, df_nor_terrain
gc.collect()


  FILE SUMMARY
  severn_terrain_ready.parquet:           (35278130, 14)
  northumbria_terrain_ready.parquet:      (21391824, 14)
  era5_severn_pixel_features.parquet:     (212, 81)
  era5_northumbria_pixel_features.parquet:(119, 81)

  CHECK 1: Null counts
  Severn terrain: 0 nulls (PASS)
  Northumbria terrain: 0 nulls (PASS)
  Severn ERA5: 0 nulls (PASS)
  Northumbria ERA5: 0 nulls (PASS)

  CHECK 2: Target class distribution

  SEVERN (train):
    risk_0_2m: 0:90.6%, 1:2.4%, 2:2.2%, 3:1.2%, 4:3.6%
    risk_0_3m: 0:90.7%, 1:3.0%, 2:1.9%, 3:1.1%, 4:3.3%
    risk_0_6m: 0:90.8%, 1:4.2%, 2:1.5%, 3:0.9%, 4:2.6%
    risk_0_9m: 0:90.9%, 1:5.1%, 2:1.2%, 3:0.7%, 4:2.1%
    risk_1_2m: 0:91.3%, 1:5.5%, 2:1.0%, 3:0.5%, 4:1.7%

  NORTHUMBRIA (test):
    risk_0_2m: 0:93.7%, 1:1.2%, 2:1.8%, 3:1.0%, 4:2.4%
    risk_0_3m: 0:93.7%, 1:1.5%, 2:1.7%, 3:0.9%, 4:2.2%
    risk_0_6m: 0:93.9%, 1:2.3%, 2:1.4%, 3:0.7%, 4:1.7%
    risk_0_9m: 0:93.9%, 1:3.0%, 2:1.1%, 3:0.6%, 4:1.4%
    risk_1_2m: 0:93.9%, 1:3.5%, 

0